# Day 018 Project — Token Dashboard Chatbot

Wrap `TokenAwareChatbot` in an interactive REPL that prints token stats after every turn and a cost summary when the session ends.

## Requirements

1. Use `TokenAwareChatbot` from the exercises as your base
2. After every bot reply, print the per-turn token counts and tokens-per-second
3. When the user types `/quit`, print `chatbot.summary()` and the `cost_estimate` for gpt-4o
4. Support `/quit` to exit
5. Run the scripted checks below before submitting

## Example Output

```
You: What is the capital of France?
Bot: The capital of France is Paris.
[Tokens: 45 in / 8 out | 32.4 tok/s]

You: /quit
Calls: 1 | Input: 45 tok | Output: 8 tok | Total: 53 tok
If using gpt-4o: $0.000193
```

In [ ]:
import ollama

def extract_usage(response: dict) -> dict:
    return {
        'input_tokens':  response.get('prompt_eval_count', 0),
        'output_tokens': response.get('eval_count', 0),
        'duration_ms':   response.get('eval_duration', 0) // 1_000_000,
    }

def tokens_per_second(response: dict) -> float:
    duration_s = response.get('eval_duration', 0) / 1e9
    if duration_s == 0:
        return 0.0
    return response.get('eval_count', 0) / duration_s

class UsageTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.call_count = 0

    @property
    def total_tokens(self) -> int:
        return self.total_input + self.total_output

    def record(self, response: dict) -> None:
        usage = extract_usage(response)
        self.total_input  += usage['input_tokens']
        self.total_output += usage['output_tokens']
        self.call_count   += 1

PRICES = {
    'gpt-4o':            {'input': 2.50,  'output': 10.00},
    'claude-3-5-sonnet': {'input': 3.00,  'output': 15.00},
    'gemini-1.5-pro':    {'input': 1.25,  'output': 5.00},
}

def cost_estimate(input_tokens, output_tokens, model='gpt-4o'):
    if model not in PRICES:
        raise ValueError(f'Unknown model {model!r}. Choose from: {list(PRICES)}')
    p = PRICES[model]
    input_cost  = input_tokens  * p['input']  / 1_000_000
    output_cost = output_tokens * p['output'] / 1_000_000
    return {
        'model':       model,
        'input_cost':  round(input_cost,  6),
        'output_cost': round(output_cost, 6),
        'total_cost':  round(input_cost + output_cost, 6),
    }

class TokenAwareChatbot:
    def __init__(self, model='llama3.2',
                 system_prompt='You are a helpful assistant.'):
        self.model = model
        self.tracker = UsageTracker()
        self._history = [{'role': 'system', 'content': system_prompt}]
        self._last_response = None

    def chat(self, user_input: str) -> str:
        self._history.append({'role': 'user', 'content': user_input})
        response = ollama.chat(model=self.model, messages=self._history)
        self._last_response = response
        self.tracker.record(response)
        reply = response['message']['content']
        self._history.append({'role': 'assistant', 'content': reply})
        return reply

    def summary(self) -> str:
        return (
            f'Calls: {self.tracker.call_count} | '
            f'Input: {self.tracker.total_input} tok | '
            f'Output: {self.tracker.total_output} tok | '
            f'Total: {self.tracker.total_tokens} tok'
        )


## Your REPL Implementation

In [ ]:
SYSTEM_PROMPT = 'You are a helpful assistant. Be concise.'

def run_token_dashboard(model='llama3.2', system_prompt=SYSTEM_PROMPT):
    """
    Interactive token dashboard chatbot.
    Prints per-turn token stats after each reply.
    On /quit: prints summary() and gpt-4o cost estimate.
    """
    # TODO: create TokenAwareChatbot
    # TODO: loop: input -> call chatbot.chat -> print reply + token stats
    # TODO: on /quit: print summary and cost_estimate
    pass


## Checks

In [ ]:
import io, sys

def _run_checks():
    total = 5
    passed = 0

    # Check 1: all required names defined
    try:
        for name in ('extract_usage', 'tokens_per_second', 'UsageTracker',
                     'cost_estimate', 'TokenAwareChatbot'):
            assert name in globals(), f'{name} not defined'
        passed += 1; print('✅ Check 1: all required components defined')
    except Exception as e:
        print(f'❌ Check 1: {e}')

    # Check 2: chat() returns a string (one real Ollama call)
    try:
        bot = TokenAwareChatbot(model='llama3.2')
        reply = bot.chat('Reply with only the word yes.')
        assert isinstance(reply, str) and len(reply) > 0
        passed += 1; print('✅ Check 2: TokenAwareChatbot.chat() returns a string')
    except Exception as e:
        print(f'❌ Check 2: chat() — {e}')

    # Check 3: tracker records usage
    try:
        assert bot.tracker.call_count == 1
        assert bot.tracker.total_tokens > 0
        passed += 1; print('✅ Check 3: UsageTracker accumulates correctly')
    except Exception as e:
        print(f'❌ Check 3: tracker — {e}')

    # Check 4: summary() contains useful info
    try:
        s = bot.summary()
        assert isinstance(s, str) and len(s) > 0
        assert any(w in s.lower() for w in ('token', 'tok'))
        passed += 1; print('✅ Check 4: summary() returns token info string')
    except Exception as e:
        print(f'❌ Check 4: summary() — {e}')

    # Check 5: cost_estimate integrates with tracker
    try:
        est = cost_estimate(bot.tracker.total_input, bot.tracker.total_output, 'gpt-4o')
        assert 'total_cost' in est and est['total_cost'] >= 0
        tps = tokens_per_second(bot._last_response)
        assert isinstance(tps, float)
        passed += 1; print('✅ Check 5: cost_estimate and tokens_per_second integrate correctly')
    except Exception as e:
        print(f'❌ Check 5: integration — {e}')

    if passed == total:
        print('🎉 Project complete!')
    print(f'\nScore: {passed}/{total}')

_run_checks()


## Bonus Challenges

- Support `/stats` command that prints `chatbot.summary()` mid-session without quitting
- Show the cost estimate for all three models (gpt-4o, claude-3-5-sonnet, gemini-1.5-pro) on exit
- Add a rolling average tokens-per-second across all turns
- Add `options={'num_predict': 100}` to `ollama.chat` to cap reply length and observe the effect on output tokens
- Track how much context grows: print `prompt_eval_count` per turn and watch it increase